# Amino Acid Substitution Heatmap

Generates a heatmap of mean SGE fitness scores per amino acid substitution and position, with optional VEP predictor rows (AlphaMissense, REVEL, CADD), deletion track, and domain/exon annotations.

**Required data:** `scores` and `thresholds` sheets from the pipeline output Excel file (`*_data.xlsx`). The `scores` sheet must contain an `amino_acid_change` column.

**Optional:**
- `domains_path`: path to a domains Excel file (columns: `region_name`, `aa_residues` as `start-end`; optional `color`, `tier` columns).
- `protein_length`: set if the data does not cover the full protein.

**Required packages:** `sge-utils` (SimpleSGEViz) installed in the active environment.

**Saving:** PNG/SVG require `vl-convert-python` (`pip install vl-convert-python`). Change the extension to `.html` for interactive output (no extra package needed).

In [ ]:
import pandas as pd
from sgeviz.figures import aa_heatmap

In [ ]:
# --- Configuration ---
excel_path = "GENE_data.xlsx"     # path to pipeline output Excel file
gene = "GENE"                     # gene name for plot title
protein_length = None             # set to integer (e.g. 777) if known; None = inferred from data
domains_path = None               # set to path string of a domains .xlsx file, or None
px_per_aa = 4                     # pixels per amino acid column (reduce for wide proteins)
output_path = f"{gene}_aa_heatmap.png"  # change to .html for interactive output, or .svg

# --- Plot customization (optional) ---
score_domain = (-0.2, 0)    # fitness score color scale range (min, max); clamp=True so out-of-range values take the endpoint color
height_per_row = 25         # height in pixels per amino acid row

In [ ]:
# --- Load data ---
scores_df = pd.read_excel(excel_path, sheet_name="scores")
thresh_df = pd.read_excel(excel_path, sheet_name="thresholds")

thresholds = [
    thresh_df["non_functional_threshold"].iloc[0],
    thresh_df["functional_threshold"].iloc[0],
]

if "amino_acid_change" not in scores_df.columns:
    raise ValueError("No 'amino_acid_change' column found in scores sheet — heatmap requires amino acid annotations.")

print(f"Loaded {len(scores_df)} variants. Thresholds: {thresholds}")

In [ ]:
# --- Generate plot ---
from pathlib import Path

chart = aa_heatmap.make_plot(
    scores_df,
    gene=gene,
    thresholds=thresholds,
    protein_length=protein_length,
    domains_path=Path(domains_path) if domains_path else None,
    px_per_aa=px_per_aa,
    score_domain=score_domain,
    height_per_row=height_per_row,
)
chart

In [ ]:
# --- Save ---
chart.save(output_path)
print(f"Saved: {output_path}")